In this notebook I will be taking the two databases I have and joining them using SQL in this notebook. I will be using duckdb library to be able to use python.

In [1]:
import duckdb
import pandas as pd
import numpy as np

In [2]:
# Connect to an in-memory DuckDB database
con = duckdb.connect(database=':memory:')

In [3]:
# DuckDB can read straight from the CSV files
result = con.execute("""
    SELECT *
    FROM 'train_transaction.csv'
    LIMIT 5
""").df()  # .df() converts result to a pandas DataFrame

result

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [4]:
joined = con.execute("""
    SELECT t.*, i.*
    FROM 'train_transaction.csv' t
    LEFT JOIN 'train_identity.csv' i
        ON t.TransactionID = i.TransactionID
""").df()

joined.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(590540, 435)

In [5]:
# register it as a persistent table so you're not re-reading the CSV every time
con.execute("""
    CREATE TABLE train_joined AS
    SELECT t.*, i.*
    FROM 'train_transaction.csv' t
    LEFT JOIN 'train_identity.csv' i
        ON t.TransactionID = i.TransactionID
""")

# Now you can just query the table
con.execute("SELECT COUNT(*) FROM train_joined").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,count_star()
0,590540


In [6]:
con.execute("""
    COPY train_joined TO 'train_joined.csv' (HEADER, DELIMITER ',')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Above I have made a new phsical csv do we dont have to run this process every time. 